# Exportar el clasificador de triage a JSON

**Meta:** entrenar la regresión logística (LR) y guardar su "cerebro" (los pesos) en un JSON que el *worker* (JavaScript) pueda usar **sin Python**.

👉 Corré las celdas **de arriba hacia abajo**, en orden.

## 0 · Setup — pararse en la raíz del repo

El notebook necesita el directorio raíz del proyecto para encontrar `src/` y `data/`.

In [ ]:
import os
os.chdir("/Users/vladislavmarinovich/PycharmProjects/saas-support-rag-triage")
print("cwd:", os.getcwd())
print("¿está el dataset?:", os.path.exists("data/tickets_v2.jsonl"))
print("¿está el cache?:", os.path.exists("data/ticket_features_v2.npy"))

## 1 · Entrenar la LR y mirar sus piezas

Entrenamos la LR sobre TODA la data y *miramos* qué guarda adentro. Todavía no exportamos.

### Por qué
La LR vive en Python (scikit-learn). El worker en vivo es JavaScript → no puede correr sklearn.
La solución no es montar un servidor Python (over-engineering), sino **llevar los pesos a JS**.
Una LR entrenada es solo números: `logits = W · x + b → argmax`. Eso es un dot product,
justo lo que el worker ya hace para el cosine. Y `x` = el embedding que ya calcula para retrieval.

### Las 3 piezas que definen el "cerebro" de la LR
| Objeto | Qué es | Forma esperada |
|---|---|---|
| `coef_` | matriz de pesos **W** — una fila por clase | `(n_clases, 768)` |
| `intercept_` | sesgo **b** — uno por clase | `(n_clases,)` |
| `classes_` | nombres de las clases, **en orden** | lista |

⚠️ El orden de `classes_` es sagrado: en JS, `argmax(logits)` da un índice → `classes_[índice]` es la etiqueta. Si se desalinea, predice mal.

### Gotcha crítico (para que el test pase después)
La LR se entrena sobre embeddings **crudos** (ver `features.py`, no normaliza).
El worker, para el cosine, **sí** normaliza. → En JS hay que alimentar la LR con el embedding **crudo**, no el normalizado. Mantené los dos.

In [ ]:
# Entrena la LR y muestra las piezas de cada etiqueta.
from src.features import build_features            # trae X (embeddings) + tickets
from sklearn.linear_model import LogisticRegression

# Las 5 etiquetas que predecimos por ticket
LABELS = ["topic", "type", "priority", "routing", "sentiment"]

# X = matriz (n, 768): un embedding por ticket.  tickets = lista con sus etiquetas.
X, tickets = build_features()
print("Tickets:", len(tickets), "| dims por embedding:", X.shape[1])

for label in LABELS:
    y = [t[label] for t in tickets]               # la respuesta correcta de cada ticket
    clf = LogisticRegression(max_iter=1000, class_weight="balanced")
    clf.fit(X, y)                                 # aprende de TODOS los tickets
    # coef_ = pesos (W) · intercept_ = sesgo (b) · classes_ = nombres de clases
    print(label, "->", clf.coef_.shape, clf.intercept_.shape, list(clf.classes_))

### 1b · La misma inspección, en tabla

Más cómodo de leer. Necesita que la celda anterior ya haya definido `LABELS`, `X` y `tickets`.

In [ ]:
import pandas as pd

rows = []
for label in LABELS:
    y = [t[label] for t in tickets]
    clf = LogisticRegression(max_iter=1000, class_weight="balanced").fit(X, y)
    rows.append({
        "etiqueta": label,
        "n_clases": len(clf.classes_),
        "clases": ", ".join(map(str, clf.classes_)),
    })

pd.DataFrame(rows)

## 2 · Exportar los pesos a JSON

Volcamos `coef_` (W), `intercept_` (b) y `classes_` de cada etiqueta a `worker/classifier_weights.json`,
listo para bundlear en el worker. `.tolist()` convierte los arrays de numpy en listas de Python
(porque JSON no sabe guardar tipos numpy).

In [ ]:
import json

WEIGHTS = {}  # acá guardamos el cerebro de cada etiqueta

for label in LABELS:
    y = [t[label] for t in tickets]
    clf = LogisticRegression(max_iter=1000, class_weight="balanced")
    clf.fit(X, y)
    WEIGHTS[label] = {
        "classes":   clf.classes_.tolist(),    # nombres, EN ORDEN
        "coef":      clf.coef_.tolist(),        # W: [n_clases][768]
        "intercept": clf.intercept_.tolist(),   # b: [n_clases]
    }

# un solo JSON con las 5 etiquetas, al lado del worker para bundlearlo
with open("worker/classifier_weights.json", "w") as f:
    json.dump(WEIGHTS, f)

print("KB de pesos:", round(os.path.getsize("worker/classifier_weights.json")/1024, 1), "KB")

## Qué sigue

**Paso 3:** inferencia en JS dentro del worker — `logits = W·x + b → argmax`, reutilizando el embedding **crudo**.
**Test clave (verificación):** que la predicción en JS coincida con la de Python sobre una muestra de tickets.